In [1]:
#CREACION DE CARPETAS
!mkdir -p datos/crudos
!mkdir -p datos/procesados
!mkdir -p datos/externos
print("Carpetas creadas !!!!!!")

Carpetas creadas !!!!!!


In [ ]:
#IMPORTACION DE LIBRERIAS
import pandas as pd #Manejar datos como excel
import sqlite3 #Para base de datos ligera
import json #Abrir y manipular archivos json
from pathlib import Path #Atajos
from datetime import time #Con
import numpy as np
print("Libreria importada con exito!!")

In [ ]:
#Rutas de trabajo ya definidas "atajos"
BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR/"datos"/"crudos"
DATA_EXTERNAL = BASE_DIR/"datos"/"externos"
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
print(f"Datos crudos-> {DATA_RAW}")
print(f"Datos procesados-> {DATA_PROCESSED}")
print(f"Datos externos -> {DATA_EXTERNAL}")
DOCUMENTOS = BASE_DIR/"datos"/"documentos"
DOCUMENTOS.mkdir(parents=True, exist_ok=True)
print(f"Carpetas creadas con exito en {DOCUMENTOS}")

In [ ]:
#CARGANDO DATASET DE LA PARTE UNO
ruta_entrada = DATA_PROCESSED / "dataset_integrado.csv"
df = pd.read_csv(ruta_entrada)
print(f"Cargado con exito en {ruta_entrada}")
print(f"Dimensiones: {df.shape[0]} Columnas: {df.shape[1]}")
df.head(5)

In [ ]:
# CREAR COPIAS DE TRABAJO
df_original = df.copy()
df_limpio = df.copy()
print("COPIAS DE TRABAJO CREADAS !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
print(f"Original: {df_original.shape}")
print(f"Para limpiar{df_limpio.shape}")

In [ ]:
#DIAGNOSTICO INICIAL DE DATOS
#DIMENSIONES
print("Dimensiones:===================================")
print(f"Filas:{df_limpio.shape[0]} ,Columnas: {df_limpio.shape[1]}\n")
#¿Estaran bien escritos?
print("Nombres de columnas ")
print(df_limpio.columns.tolist())

In [ ]:
#TIPOS DE DATOS - ¿CADA COLUMNA TENDRÁ EL TIPO CORRECTO?
print("Tipos de datos===========")
print(df_limpio.dtypes)

In [ ]:
#INFORMACION GENERAL
print("Informacion general")
df_limpio.info()

In [ ]:
#IDENTIFICAR VALORES NULOS - ¿HABRÁN DATOS FALTANTES?
print("Valores nulos")
nulos = df_limpio.isnull().sum()
print(nulos)
#PORCENTJE DE NULOS
print("PORCENTAJE-----------------")
porcentaje_nulos = (df_limpio.isnull().mean() * 100).round(2)
print(porcentaje_nulos)

In [ ]:
#DUPLICADOS - - ¿ExÍsten filas que repiten?
print("\nRegistros duplicados")
duplicados = df_limpio.duplicated().sum()
print('Registros duplicados:', duplicados)

In [ ]:
#VALORES UNICOS ¿Poséen valores consistentes?
print("VALORES UNICOS")
columnas_categoricas = df_limpio.select_dtypes(include='object').columns

for columna in columnas_categoricas:
 print(' Columna:', columna)
 print(df_limpio[columna].unique())


In [ ]:
#LIMPIEZA DE COLUMNAS
print("Columnas antes_____")
print(df_limpio.columns.tolist())

df_limpio.columns = (
    df_limpio.columns
    .str.strip() #ELIMINA ESPACIOS AL INICIO Y AL FINAL
    .str.lower() #CONVIERTE A MINUSCULAS
    .str.replace(' ', '_', regex=False)
)
print("COLUMNAS DESPUES__________")
print(df_limpio.columns.tolist())

In [ ]:
#ELIMINAR ESPACIOS EN TEXTOS
#TODAS LAS COLUMNAS DEBEN TENER SUS VALORES LIMPIO
columnas_texto = df_limpio.select_dtypes(include='object').columns
print(f"Columnas de texto: {columnas_texto.tolist()}")
for columna in columnas_texto:
  df_limpio[columna] = (df_limpio[columna].astype('string').str.strip())
print("ESPACIOS ELIMINADOS")

In [ ]:
#No realizamos la normalizacion y el mapeo de ninguna columna debido
#a que como grupo analizamos y llegamos a que no tenia
#nada que corregir en esta parte, por que todos los datos unicos
#no contenian niguna variante mal escrita.

In [ ]:
#Convertir los "falsos nulos" a nulos reales (np.nan) en TODAS las columnas a la vez
textos_vacios = ['nan', 'NaN', '<NA>', 'None', 'NaT','','nat']
df_limpio = df_limpio.replace(textos_vacios, np.nan)

#Hacer el conteo, pero mostrando solo las columnas que tienen valores nulos
print("Reporte de valores nulos reales:")
nulos_por_columna = df_limpio.isnull().sum()

for columna, cantidad in nulos_por_columna.items():
    if cantidad > 0:  # Solo imprime si la columna tiene al menos 1 nulo
        print(f"{columna}: {cantidad}")


In [ ]:
#COMPLETAR COLUMNAS AUSENTES

# Columna edad_repartidor
df_limpio['edad_repartidor'] = df_limpio['edad_repartidor'].fillna(df_limpio['edad_repartidor'].median())

# Columna calificacion_repartidor
df_limpio['calificacion_repartidor'] = df_limpio['calificacion_repartidor'].fillna(df_limpio['calificacion_repartidor'].mean().round(1))

# Columna hora_pedido
df_limpio['hora_pedido'] = df_limpio['hora_pedido'].fillna(df_limpio['hora_pedido'].mode()[0])

# Columna condiciones_climaticas
df_limpio['condiciones_climaticas'] = df_limpio['condiciones_climaticas'].fillna('Desconocido')

# Columna densidad_trafico
df_limpio['densidad_trafico'] = df_limpio['densidad_trafico'].fillna('Desconocido')

# Columna festival
df_limpio['festival'] = df_limpio['festival'].fillna('No')

# Columna ciudad
df_limpio['ciudad'] = df_limpio['ciudad'].fillna('Desconocido')

# Columna multiples_entregas
# lo más razonable es asumir 0 (ninguna entrega múltiple registrada)
df_limpio['multiples_entregas'] = df_limpio['multiples_entregas'].fillna(0)

#Columna fecha
# Encuentra la fecha que más se repite y la usa para rellenar
fecha_mas_comun = df_limpio['fecha_pedido'].mode()[0]
df_limpio['fecha_pedido'] = df_limpio['fecha_pedido'].fillna(fecha_mas_comun)
print(df_limpio.isnull())
print("Columnas completadas:")
print(df_limpio[['edad_repartidor', 'calificacion_repartidor', 'hora_pedido',
                  'condiciones_climaticas', 'densidad_trafico', 'festival',
                  'ciudad', 'multiples_entregas']].head(7))
nulos = df_limpio.isnull().sum()
print(nulos)

In [ ]:
# ELIMINAR DUPLICADOS
# Eliminamos filas repetidas exactas

print(f"Duplicados antes: {df_limpio.duplicated().sum()}")
df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
print(f"Duplicados después: {df_limpio.duplicated().sum()}")
print(f"Nuevas dimensiones: {df_limpio.shape}")

In [ ]:
# CONVERSION DE DATOS
df_limpio['edad_repartidor'] = df_limpio['edad_repartidor'].astype(int)
print("EDAD REPARTIDOR convertida:", df_limpio['edad_repartidor'].dtype)

df_limpio['multiples_entregas'] = df_limpio['multiples_entregas'].astype(int)
print("MULTIPLES ENTREGAS convertida:", df_limpio['multiples_entregas'].dtype)

# CONVERSION DE FECHA
df_limpio["fecha_pedido"] = pd.to_datetime(
    df_limpio["fecha_pedido"],
    format="mixed",
    errors="coerce"
)

# FUNCION PARA NORMALIZAR LAS HORAS
def convertir_hora(valor):
    # Horas expresadas como fraccion de un dia
    try:
        numero = float(valor)
        if 0 <= numero <= 1:
            segundos = round(numero * 86400) % 86400
            return pd.to_timedelta(segundos, unit="s")
    except ValueError:
        pass

    # Horas expresadas como texto
    hora = pd.to_timedelta(valor, errors="coerce")

    return hora

def convertir_hora(valor):
    if pd.isna(valor):
        return pd.NaT

    texto = str(valor).strip()

    # Caso 1: hora expresada como fracción del día
    try:
        numero = float(texto)
        if 0 <= numero <= 1:
            segundos = round(numero * 86400) % 86400
            horas, resto = divmod(segundos, 3600)
            minutos, segundos = divmod(resto, 60)
            return time(horas, minutos, segundos)
    except (ValueError, TypeError):
        pass

    # Caso 2: hora en formato HH:MM o HH:MM:SS
    try:
        partes = texto.split(":")
        if len(partes) in (2, 3):
            horas = int(partes[0])
            minutos = int(partes[1])
            segundos = float(partes[2]) if len(partes) == 3 else 0

            total = round(horas * 3600 + minutos * 60 + segundos) % 86400
            horas, resto = divmod(total, 3600)
            minutos, segundos = divmod(resto, 60)

            return time(horas, minutos, segundos)
    except (ValueError, TypeError):
        pass

    return pd.NaT


# Convertir las columnas
df_limpio["hora_pedido"] = df_limpio["hora_pedido"].apply(convertir_hora)
df_limpio["hora_recogida_pedido"] = (
    df_limpio["hora_recogida_pedido"].apply(convertir_hora)
)

# Comprobar los resultados
print("Fecha:", df_limpio["fecha_pedido"].dtype)
print("Hora del pedido:", df_limpio["hora_pedido"].dtype)
print("Hora de recogida:", df_limpio["hora_recogida_pedido"].dtype)

print(
    df_limpio[
        ["fecha_pedido", "hora_pedido", "hora_recogida_pedido"]
    ].head(10)
)

nulos = df_limpio.isnull().sum()
print(nulos)

In [ ]:
# CONVERSION DE FECHA (Formato YYYY-MM-DD)
df_limpio["fecha_pedido"] = pd.to_datetime(
    df_limpio["fecha_pedido"],
    format="mixed",
    errors="coerce"
)

In [ ]:
# CONVERTIR TIPOS DE DATOS A INT

df_limpio['edad_repartidor'] = df_limpio['edad_repartidor'].astype(int)
print("EDAD REPARTIDOR convertida:", df_limpio['edad_repartidor'].dtype)

df_limpio['multiples_entregas'] = df_limpio['multiples_entregas'].astype(int)
print("MULTIPLES ENTREGAS convertida:", df_limpio['multiples_entregas'].dtype)

In [ ]:
# REPORTE DE CALIDAD

reporte_calidad = pd.DataFrame({
    'columna': df_limpio.columns,
    'tipo_dato': [str(df_limpio[col].dtype) for col in df_limpio.columns],
    'valores_nulos': [df_limpio[col].isnull().sum() for col in df_limpio.columns],
    'valores_unicos': [df_limpio[col].nunique(dropna=False) for col in df_limpio.columns]
})

print("REPORTE DE CALIDAD:")
display(reporte_calidad)

In [ ]:
# CREACION DEL PRECIO DEL PEDIDO

# Precio base según el tipo de pedido
precios_base = {
    "Snack": 3.50,
    "Drinks": 4.60,
    "Meal": 6.75,
    "Buffet": 13.99
}

# Asignar precio base según el tipo de pedido
df_limpio["precio_base"] = df_limpio["tipo_pedido"].map(precios_base)

# Tratar los valores nulos de entregas múltiples como 0
df_limpio["multiples_entregas"] = (
    df_limpio["multiples_entregas"].fillna(0)
)

# Calcular el precio final según las entregas adicionales
df_limpio["precio_pedido"] = (
    df_limpio["precio_base"] *
    (1 + df_limpio["multiples_entregas"])
).round(2)

print("Precio de los pedidos calculado correctamente.")

print(
    df_limpio[
        [
            "tipo_pedido",
            "multiples_entregas",
            "precio_base",
            "precio_pedido"
        ]
    ].head(10)
)

In [ ]:
#No podemos aplcar ese metodo numerico ya que este data set se trata sobre calcular el timepo del pedido

In [ ]:
# CORREGIR VALORES NUMERICOS NEGATIVOS
# En este caso, asumimos que es un error de signo y lo corregimos con valor absoluto
#Edad
edades_negativos = df_limpio[df_limpio['edad_repartidor'] < 0]
print(f"Edades negativos encontrados: {len(edades_negativos)}")
if len(edades_negativos) > 0:
    print(edades_negativos[['id', 'edad_repartidor']])
df_limpio['edad_repartidor'] = df_limpio['edad_repartidor'].abs()

#Calificacion
calificacion_negativos = df_limpio[df_limpio['calificacion_repartidor'] < 0]
print(f"Calificaciones negativos encontrados: {len(calificacion_negativos)}")
if len(calificacion_negativos) > 0:
    print(calificacion_negativos[['id', 'calificacion_repartidor']])
df_limpio['calificacion_repartidor'] = df_limpio['calificacion_repartidor'].abs()

#Tiempo de entrega
tiempo_negativos = df_limpio[df_limpio['tiempo_entrega_min'] < 0]
print(f"Tiempo de entrega negativos encontrados: {len(tiempo_negativos)}")
if len(tiempo_negativos) > 0:
    print(tiempo_negativos[['id', 'tiempo_entrega_min']])
df_limpio['tiempo_entrega_min'] = df_limpio['tiempo_entrega_min'].abs()


print("Datos corregidos")

In [ ]:
# CARACTERÍSTICAS TEMPORALES
# Extraemos año, mes, día y día de semana de la fecha

df_limpio['anio'] = df_limpio['fecha_pedido'].dt.year
df_limpio['mes'] = df_limpio['fecha_pedido'].dt.month
df_limpio['dia'] = df_limpio['fecha_pedido'].dt.day
df_limpio['dia_semana'] = df_limpio['fecha_pedido'].dt.day_name()
df_limpio['es_fin_semana'] = df_limpio['fecha_pedido'].dt.dayofweek >= 5

print("CARACTERÍSTICAS TEMPORALES CREADAS:")
df_limpio[['fecha_pedido', 'anio', 'mes', 'dia', 'dia_semana', 'es_fin_semana']].head()

In [ ]:
# REPORTE DE CALIDAD
# Tabla resumen de cada columna

reporte_calidad = pd.DataFrame({
    'columna': df_limpio.columns,
    'tipo_dato': [str(df_limpio[col].dtype) for col in df_limpio.columns],
    'valores_nulos': [df_limpio[col].isnull().sum() for col in df_limpio.columns],
    'valores_unicos': [df_limpio[col].nunique(dropna=False) for col in df_limpio.columns]
})

print("REPORTE DE CALIDAD:")
reporte_calidad

In [ ]:
# GUARDAR REPORTE EN EXCEL
ruta_reporte = DOCUMENTOS / "reporte_calidad.xlsx"
reporte_calidad.to_excel(ruta_reporte, index=False)
print(f"Reporte guardado: {ruta_reporte}")

In [ ]:
# GUARDAR DATASET LIMPIO
ruta_salida = DATA_PROCESSED / "dataset_limpio.csv"
df_limpio.to_csv(ruta_salida, index=False, encoding='utf-8')
print(f"Dataset limpio guardado: {ruta_salida}")